In [3]:
import yfinance as yf
import pandas as pd
import numpy as np

In [23]:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "META", "NVDA"]
ohlcv_data = {}
for ticker in tickers: 
    temp = yf.download(ticker, period='1y', interval='1d')
    temp.dropna(how='any', inplace=True)
    ohlcv_data[ticker] = temp

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [33]:
def CAGR(DF):
    df = DF.copy()
    df["return"] = df["Close"].pct_change()
    df["cum_return"] = (1+df["return"]).cumprod()
    n = len(df)/252
    CAGR = (df["cum_return"].iloc[-1])**(1/n) - 1
    return CAGR * 100

def calculate_volatility(DF):
    df = DF.copy()
    df["return"] = df["Close"].pct_change()
    vol = df["return"].std() * np.sqrt(252)
    return vol * 100

def sharpe_ratio(DF, risk_free_rate=0.03):
    df = DF.copy()
    return (CAGR(df) - risk_free_rate) / calculate_volatility(df)

def sortino_ratio(DF, risk_free_rate=0.03):
    df = DF.copy()
    df.columns = df.columns.droplevel(1)
    df["return"] = df["Close"].pct_change()
    neg_return = np.where(df["return"] > 0, 0, df["return"])
    neg_vol = pd.Series(neg_return[neg_return !=0]).std() * np.sqrt(252)
    cagr = CAGR(df) / 100
    return (cagr - risk_free_rate) / neg_vol

In [34]:
for ticker in ohlcv_data:
    sharp = sharpe_ratio(ohlcv_data[ticker])
    sortino = sortino_ratio(ohlcv_data[ticker])
    print(f"Sharpe Ratio {ticker} : {sharp}")
    print(f"Sortino Ratio {ticker} : {sortino}")

Sharpe Ratio AAPL : 2.4459799678552447
Sortino Ratio AAPL : 3.5396769347053905
Sharpe Ratio MSFT : -0.8099944600712853
Sortino Ratio MSFT : -1.2223392959506394
Sharpe Ratio GOOGL : 2.890818696882792
Sortino Ratio GOOGL : 5.325135845248847
Sharpe Ratio AMZN : 0.3404499516221002
Sortino Ratio AMZN : 0.3670112134083425
Sharpe Ratio TSLA : 0.43501767278350034
Sortino Ratio TSLA : 0.6135269308565261
Sharpe Ratio META : -0.19153820022434756
Sortino Ratio META : -0.39203992307077695
Sharpe Ratio NVDA : 0.4924329033416539
Sortino Ratio NVDA : 0.670216738045237
